In [ ]:

# -*- coding: utf-8 -*-
"""
第一部分：自适应 EMD 二分类数据预处理 (原汁原味二分类标签版)
"""
import os
# 严格锁死全局随机种子
os.environ['PYTHONHASHSEED'] = '42'
import random
import numpy as np
random.seed(2024)
np.random.seed(2024)

import pandas as pd
import matplotlib.pyplot as plt
from PyEMD import EMD
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

class AdaptiveEMDProcessor:
    def __init__(self, n_imfs=8, energy_threshold=0.02, corr_threshold=0.12, min_corr=0.8):
        self.n_imfs = n_imfs
        self.emd = EMD()
        self.energy_threshold = energy_threshold 
        self.corr_threshold = corr_threshold      
        self.min_corr = min_corr                  

    def _calculate_imf_metrics(self, imfs, orig_signal):
        total_energy = np.sum(orig_signal ** 2)
        energy_ratios = []
        correlations = []
        
        for imf in imfs:
            imf_energy = np.sum(imf ** 2)
            energy_ratios.append(imf_energy / total_energy)
            
            if np.std(imf) < 1e-6 or np.std(orig_signal) < 1e-6:
                correlations.append(0)
            else:
                corr = np.corrcoef(orig_signal, imf)[0, 1]
                correlations.append(corr)
        return np.array(energy_ratios), np.array(correlations)
    
    def _calc_correlation(self, orig, recon):
        min_len = min(len(orig), len(recon))
        if min_len < 2: return 0.0
        orig, recon = orig[:min_len], recon[:min_len]
        if np.std(orig) < 1e-6 or np.std(recon) < 1e-6: return 0.0
        return np.corrcoef(orig, recon)[0, 1]

    def _adaptive_denoise(self, sample, is_theft_sample=False):
        orig_signal = sample.copy()
        try:
            imfs = self.emd(orig_signal, max_imf=self.n_imfs)
            residue = orig_signal - np.sum(imfs, axis=0)
            if len(imfs) < 2: return orig_signal
            
            energy_ratios, correlations = self._calculate_imf_metrics(imfs, orig_signal)
            
            # 第一步：原始逻辑筛选
            if is_theft_sample:
                useful_mask = (energy_ratios > self.energy_threshold*0.8) | (np.abs(correlations) > self.corr_threshold*0.8)
            else:
                useful_mask = (energy_ratios > self.energy_threshold) | (np.abs(correlations) > self.corr_threshold)
            useful_imfs = imfs[useful_mask]
            
            # 过滤逻辑
            if not is_theft_sample and len(useful_imfs) > 1:
                useful_imfs = useful_imfs[1:]
            elif is_theft_sample and len(useful_imfs) > 1:
                first_imf_energy = np.sum(useful_imfs[0]**2) / np.sum(orig_signal**2)
                if first_imf_energy < 0.015:
                    useful_imfs = useful_imfs[1:]
            
            reconstructed = np.sum(useful_imfs, axis=0) + residue
            recon_corr = self._calc_correlation(orig_signal, reconstructed)
            
            # 第二步：不达标则放宽筛选（降至50%）
            if recon_corr < self.min_corr:
                useful_mask = (energy_ratios > self.energy_threshold*0.5) | (np.abs(correlations) > self.corr_threshold*0.5)
                useful_imfs = imfs[useful_mask]
                reconstructed = np.sum(useful_imfs, axis=0) + residue
                recon_corr = self._calc_correlation(orig_signal, reconstructed)
                
                # 第三步：仍不达标则强制保留所有IMF
                if recon_corr < self.min_corr:
                    useful_imfs = imfs 
                    reconstructed = np.sum(useful_imfs, axis=0) + residue
            
            # 幅值校准
            if np.max(reconstructed) != 0 and np.max(orig_signal) != 0:
                reconstructed = reconstructed * (np.max(orig_signal) / np.max(reconstructed))
            if len(reconstructed) > len(orig_signal):
                reconstructed = reconstructed[:len(orig_signal)]
                
            return reconstructed
        except Exception:
            return orig_signal

    def process_data(self, X_data, y_data=None):
        print(f"=== 自适应 EMD 处理 {len(X_data)} 个样本 ===")
        processed_signals = []
        for sample_idx, sample in enumerate(X_data):
            if sample_idx % 100 == 0:
                print(f"处理进度: {sample_idx}/{len(X_data)}")
            
            # 恢复你的完美原版逻辑：二分类本身就是 0 和 1
            is_theft = False
            if y_data is not None and y_data[sample_idx] == 1:
                is_theft = True
            
            processed = self._adaptive_denoise(sample, is_theft_sample=is_theft)
            processed_signals.append(processed)
        return np.array(processed_signals) if y_data is None else (np.array(processed_signals), y_data)

def main():
    print("=== 第一阶段：二分类 EMD 自适应去噪分解 ===")
    
    # ！！！请务必把这里改成你真实的二分类数据集路径！！！
    file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'
    
    try:
        df = pd.read_excel(file_path)
    except Exception:
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except Exception:
            df = pd.read_csv(file_path, encoding='ANSI')

    if 'Unnamed: 0' in df.columns:
        df.drop(['Unnamed: 0'], axis=1, inplace=True)
    
    Y = df['output'].values
    X = df.drop(['output'], axis=1).values
    
    print(f"原始标签分布，正常样本(0): {np.sum(Y==0)}, 窃电样本(1): {np.sum(Y==1)}")
    
    # ✅ 严格对齐多分类设定的切分比例 0.4
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, Y, test_size=0.5, random_state=42, stratify=Y
    )
    
    processor = AdaptiveEMDProcessor(min_corr=0.8)
    X_train, y_train = processor.process_data(X_train_raw, y_train_raw)
    X_test, y_test = processor.process_data(X_test_raw, y_test_raw)
    
    # 保存结果
    output_dir = "adaptive_emd_binary_0.8011"
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "X_train.npy"), X_train)
    np.save(os.path.join(output_dir, "y_train.npy"), y_train)
    np.save(os.path.join(output_dir, "X_test.npy"), X_test)
    np.save(os.path.join(output_dir, "y_test.npy"), y_test)
    
    print(f"\n✅ EMD 分解预处理完成！纯净数据已保存至: {output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
"""
第二部分：EMD-SE-CNN 二分类网络训练与预测 (SCI 完美对齐版)
"""
import os
# 1. 严格锁死全局随机种子，确保 SCI 论文的 100% 可复现性
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
import tensorflow as tf

random.seed(2024)
np.random.seed(2024)
tf.random.set_seed(2024)

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Reshape, Multiply, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

class SEBlock(tf.keras.layers.Layer):
    """SE注意力模块（严格匹配论文 Table 1 设置，降维系数 r=3）"""
    def __init__(self, reduction=3, **kwargs):
        super(SEBlock, self).__init__(**kwargs)
        self.reduction = reduction
        
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.fc1 = Dense(
            max(self.channels // self.reduction, 4),
            activation='relu',
            use_bias=False,
            kernel_initializer='he_normal'
        )
        self.fc2 = Dense(
            self.channels,
            activation='sigmoid',
            use_bias=False,
            kernel_initializer='he_normal'
        )
        super(SEBlock, self).build(input_shape)
        
    def call(self, inputs):
        se = GlobalAveragePooling1D()(inputs)
        se = Reshape((1, self.channels))(se)
        se = self.fc1(se)
        se = self.fc2(se)
        return Multiply()([inputs, se])
    
    def get_config(self):
        config = super(SEBlock, self).get_config()
        config.update({'reduction': self.reduction})
        return config

def main():
    print("=== 第二阶段：SCI 标准版 EMD-SE-CNN 二分类窃电检测 ===")
    
    # 1. 加载第一阶段分解好的极净数据
    data_dir = "adaptive_emd_binary_0.8011"
    try:
        X_train = np.load(os.path.join(data_dir, "X_train.npy"))
        y_train = np.load(os.path.join(data_dir, "y_train.npy"))
        X_test = np.load(os.path.join(data_dir, "X_test.npy"))
        y_test = np.load(os.path.join(data_dir, "y_test.npy"))
        
        # 增加一维通道维度，适配 Conv1D 要求的 (样本数, 时间步长, 通道数) 格式
        X_train = X_train[..., np.newaxis] 
        X_test = X_test[..., np.newaxis]   
        print(f"✅ 数据加载成功！训练集: {X_train.shape}, 测试集: {X_test.shape}")
    except FileNotFoundError:
        print("❌ 找不到数据文件夹，请先运行 01_EMD_Binary_Process.py 生成分解数据！")
        return
    
    # 2. 构建 SE-CNN 模型 (网络拓扑严格对应论文初稿)
    inputs = Input(shape=X_train.shape[1:]) 
    
    x = Conv1D(32, 5, strides=1, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(inputs)
    x = SEBlock(reduction=3)(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding='same')(x)
    x = Dropout(0.3)(x)
    
    x = Conv1D(64, 3, strides=1, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = SEBlock(reduction=3)(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding='same')(x)
    x = Dropout(0.3)(x)
    
    x = Flatten()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(64, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = Dropout(0.3)(x)
    
    # 二分类特有输出层：1 个神经元 + sigmoid
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=outputs)
    
    # 3. 编译模型 (二分类特有损失函数)
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=0.0005),
        metrics=['accuracy']
    )
    
    # 4. 训练模型
    early_stop = EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)
    
    print("\n🚀 开始模型训练...")
    history = model.fit(
        X_train, y_train,
        epochs=150,
        batch_size=32,
        validation_data=(X_test, y_test), 
        callbacks=[early_stop, reduce_lr],
        shuffle=True, # 确保训练打乱
        verbose=1
    )
    
    # ==========================================================
    # 5. 模型深度评估 (引入 SCI 窃电检测专属的 FPR, FNR, G-Mean)
    # ==========================================================
    print("\n=== 模型硬核指标评估 ===")
    # 获取预测概率 (0~1之间) 并根据 0.5 阈值转化为整数标签
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    # 计算基础分类指标
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # 计算 ROC 曲线与 AUC
    fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr_curve, tpr_curve)
    
    # 计算二分类最核心业务指标：FPR, FNR, G-Mean
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr_val = fp / (fp + tn + 1e-10) # 误报率 (把好人当贼)
    fnr_val = fn / (fn + tp + 1e-10) # 漏报率 (让贼跑了)
    g_mean = np.sqrt((tp / (tp + fn + 1e-10)) * (tn / (tn + fp + 1e-10)))
    
    # 打印学术级终端报告
    print("\n" + "="*50)
    print("=== SCI 级二分类详细性能指标 ===")
    print("="*50)
    print(f"准确率 (Accuracy)       : {accuracy:.4f}")
    print(f"精确率 (Precision)      : {precision:.4f}")
    print(f"召回率 (Recall)         : {recall:.4f}")
    print(f"F1分数 (F1-Score)       : {f1:.4f}")
    print(f"AUC面积 (ROC-AUC)       : {roc_auc:.4f}")
    print("-" * 50)
    print(f"误报率 (FPR)            : {fpr_val:.4f}  <-- (电网稽查白跑一趟的概率)")
    print(f"漏报率 (FNR)            : {fnr_val:.4f}  <-- (漏掉真正窃电贼的概率)")
    print(f"综合均值 (G-Mean)       : {g_mean:.4f}")
    print(f"马修斯系数 (MCC)        : {mcc:.4f}")
    print("="*50)
    
    print("\n分类报告:")
    print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Theft (1)'], digits=4))
    
    # 6. 导出预测结果 CSV (为了最后画二分类多模型对比图)
    save_path = "Binary_Results_CSV2"
    os.makedirs(save_path, exist_ok=True)
    
    df_save = pd.DataFrame({
        'label': y_test,
        'pred': y_pred_proba
    })
    csv_filename = os.path.join(save_path, "Binary_EMD_CNN_SE.csv")
    df_save.to_csv(csv_filename, index=False)
    print(f"\n✅ 预测结果已成功导出至: {csv_filename} (供后续画对比大图使用)")

if __name__ == "__main__":
    main()

In [ ]:
# -*- coding: utf-8 -*-
"""
【消融实验专用】CNN-SE 模型（严格对齐 EMD-CNN-SE 所有参数）
目标：证明移除 EMD 后性能的变化
"""
import os
# 1. 【核心修复】严格锁死全局随机种子，确保与主模型在同一起跑线
os.environ['PYTHONHASHSEED'] = '2024'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
import tensorflow as tf

random.seed(2024)
np.random.seed(2024)
tf.random.set_seed(2024)

import matplotlib
matplotlib.rcParams["font.family"] = ["SimHei", "Microsoft YaHei", "Arial", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Reshape, Multiply, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print("=== 消融实验：CNN-SE模型（严格对齐 EMD-CNN-SE 实验设置） ===")

# 2. 数据加载
print("\n1. 加载数据...")
file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'
df = pd.read_excel(file_path)
if 'Unnamed: 0' in df.columns:
    df.drop(['Unnamed: 0'], axis=1, inplace=True)
Y = df['output'].astype(int).values
X = df.drop(['output'], axis=1).values

# 保持归一化逻辑一致
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X).astype(np.float32)

# 3. 【核心修复】严格对齐数据集划分 (test_size=0.4, stratify=Y)
# 必须与 EMD-CNN-SE 的 0.4 保持绝对一致，否则指标无对比意义
x_train, x_test, y_train, y_test = train_test_split(
    X_scaled, Y, 
    test_size=0.5,     # ✅ 修复：从 0.2 改回 0.4
    random_state=2024,
    shuffle=True,      # ✅ 修复：开启打乱
    stratify=Y         # ✅ 修复：开启分层抽样，确保类别比例一致
)

# 调整维度
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]
input_dim = x_train.shape[1]

# 4. 构建模型 (结构与 EMD-CNN-SE 1:1 复制)
class SEBlock(tf.keras.layers.Layer):
    def __init__(self, reduction=1, **kwargs): # ✅ 默认 reduction 设为 3
        super(SEBlock, self).__init__(**kwargs)
        self.reduction = reduction
        
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.fc1 = Dense(max(self.channels // self.reduction, 4), activation='relu', use_bias=False, kernel_initializer='he_normal')
        self.fc2 = Dense(self.channels, activation='sigmoid', use_bias=False, kernel_initializer='he_normal')
        super(SEBlock, self).build(input_shape)
        
    def call(self, inputs):
        se = GlobalAveragePooling1D()(inputs)
        se = Reshape((1, self.channels))(se)
        se = self.fc1(se)
        se = self.fc2(se)
        return Multiply()([inputs, se])

inputs = Input(shape=(input_dim, 1))
x = Conv1D(32, 5, strides=1, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(inputs)
x = SEBlock(reduction=2)(x)
x = BatchNormalization()(x)
x = MaxPooling1D(2, padding='same')(x)
x = Dropout(0.4)(x)

x = Conv1D(64, 3, strides=1, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(x)
x = SEBlock(reduction=4)(x)
x = BatchNormalization()(x)
x = MaxPooling1D(2, padding='same')(x)
x = Dropout(0.4)(x)

x = Flatten()(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(2e-4))(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(64, activation='relu', kernel_regularizer=l2(2e-4))(x)
x = Dropout(0.4)(x)

outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs=inputs, outputs=outputs)

model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0005), metrics=['accuracy'])

# 5. 训练配置 (严格对齐)
early_stop = EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)

print("\n🚀 开始训练（消融实验 - 移除EMD版）...")
history = model.fit(
    x_train, y_train,
    epochs=150, batch_size=32,
    validation_data=(x_test, y_test),
    callbacks=[early_stop, reduce_lr],
    shuffle=True, verbose=1
)

# 6. 【核心修复】补齐二分类硬核指标 (FPR/FNR/G-Mean)
print("\n=== 模型深度评估 ===")
y_pred_proba = model.predict(x_test, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

# 计算各项指标
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
fpr_val = fp / (fp + tn + 1e-10)
fnr_val = fn / (fn + tp + 1e-10)
g_mean = np.sqrt((tp / (tp + fn + 1e-10)) * (tn / (tn + fp + 1e-10)))
mcc = matthews_corrcoef(y_test, y_pred)
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr_curve, tpr_curve)

print("\n" + "="*50)
print(f"准确率 (Accuracy) : {accuracy:.4f}")
print(f"精确率 (Precision): {precision:.4f}")
print(f"召回率 (Recall)   : {recall:.4f}")
print(f"F1分数 (F1-Score) : {f1:.4f}")
print(f"AUC面积 (ROC-AUC) : {roc_auc:.4f}")
print("-" * 50)
print(f"误报率 (FPR)      : {fpr_val:.4f}")
print(f"漏报率 (FNR)      : {fnr_val:.4f}")
print(f"综合均值 (G-Mean) : {g_mean:.4f}")
print(f"马修斯相关系数    : {mcc:.4f}")
print("="*50)

# 7. 保存结果供画图
save_path = r"E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\正式sci论文\二区期刊的sci论文\无泄漏重跑实验\CNN_SE"
os.makedirs(save_path, exist_ok=True)
pd.DataFrame({'label': y_test, 'pred': y_pred_proba}).to_csv(os.path.join(save_path, "Binary_CNN_SE_NoEMD.csv"), index=False)
print(f"✅ 预测结果已保存至: {save_path}")